# Batch Process Images (Fearful Expression)

This notebook runs the complete MagicFace pipeline (preprocess, retrieve_bg, and inference) on 50 images from `processed_identities` to apply a fearful expression.

In [ ]:
import os
import sys
import subprocess

PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Update PYTHON_PATH for your environment
# Windows example: 'C:/Users/faris3/AppData/Local/miniconda3/envs/magicface/python.exe'
PYTHON_PATH = 'C:/Users/faris3/AppData/Local/miniconda3/envs/magicface/python.exe'

# Source directories (relative to magic_face/)
CATEGORIES = {
    'dominant':   os.path.join('..', 'face_processing', 'Dominant', 'ProcessedDominantFaces', 'processed_identities'),
    'submissive': os.path.join('..', 'face_processing', 'Submissive', 'ProcessedSubmissiveFaces', 'processed_identities'),
}

N_IMAGES = 50  # Number of images to process per category

for category, input_dir in CATEGORIES.items():
    image_files = sorted(f for f in os.listdir(input_dir) if f.endswith('.jpg'))[:N_IMAGES]
    print(f"[{category}] Found {len(image_files)} images to process.")


In [ ]:
for category, input_dir in CATEGORIES.items():
    image_files = sorted(f for f in os.listdir(input_dir) if f.endswith('.jpg'))[:N_IMAGES]
    output_dir = os.path.join('edited_images_fear', category)
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n{'#'*50}\nCategory: {category.upper()} ({len(image_files)} images)\n{'#'*50}")

    for img_name in image_files:
        input_image = os.path.join(input_dir, img_name)
        base_name = os.path.splitext(img_name)[0]
        tmp_dir = os.path.join('test_images', category)
        os.makedirs(tmp_dir, exist_ok=True)
        cropped_image = os.path.join(tmp_dir, f"{base_name}_cropped.png")
        bg_image = os.path.join(tmp_dir, f"{base_name}_bg.png")

        print(f"\n{'='*40}\nProcessing {img_name} [{category}]...\n{'='*40}")

        # 1. Preprocess
        print("Step 1: Preprocess (Crop Face)")
        subprocess.run(f"{PYTHON_PATH} utils/preprocess.py --img_path {input_image} --save_path {cropped_image}", shell=True)

        # 2. Retrieve Background
        print("Step 2: Retrieve Background")
        subprocess.run(f"{PYTHON_PATH} utils/retrieve_bg.py --img_path {input_image} --save_path {bg_image}", shell=True)

        # 3. Inference (Fearful, closed-mouth)
        # AUs: AU1 (Inner Brow Raiser), AU2 (Outer Brow Raiser), AU4 (Brow Lowerer),
        #      AU5 (Upper Lid Raiser), AU20 (Lip Stretcher)
        # NOTE: AU25 (Lips Part) and AU26 (Jaw Drop) are intentionally excluded to keep mouths closed.
        print("Step 3: Inference (Fearful)")
        subprocess.run(
            f"{PYTHON_PATH} inference.py --img_path {cropped_image} --bg_path {bg_image} "
            f"--au_test AU1+AU2+AU4+AU5+AU20 --AU_variation 5+5+5+5+5 --saved_path {output_dir}",
            shell=True
        )

        print(f"Finished {img_name}\n")

    print(f"\n[{category}] Done. Outputs in: {output_dir}\n")

print("\nBatch processing complete!")
